# HTB Skills Assessment — Privacy-Preserving Fashion-MNIST CNN

This notebook trains the exact `FashionMNISTCNN` architecture required by the
assessment, while using privacy-oriented training choices to reduce membership
inference leakage. The server requires at least 70% test accuracy and at least
a 40% reduction from its vulnerable-model MIA baseline.

The submission control stays off by default. Run the local checks first; after
they pass, enable `SUBMIT` only for the authorized HTB target.

## 1. Configuration and stable execution

The previous Windows crashes came from native worker/CUDA failures, so this
notebook uses one data-loader worker, bounded PyTorch threads, and CPU by
default. `USE_CUDA` is an explicit opt-in rather than an automatic choice.

In [ ]:
import json
import random
from pathlib import Path
import numpy as np
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors.torch import save_file

INSTANCE_HOST = '154.57.164.82:37906'
BASE_URL = f'http://{INSTANCE_HOST}'
MODEL_PATH = Path('defended_model.safetensors')
SUBMIT = False
USE_CUDA = False
SEED = 1337
EPOCHS = 8
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 5e-4
LABEL_SMOOTHING = 0.10
INPUT_NOISE_STD = 0.05
REQUEST_TIMEOUT = 600

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device('cuda' if USE_CUDA and torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

## 2. The live baseline and the success threshold

The server gives a vulnerable baseline MIA advantage. If that value is `a`,
a 40% reduction means the defended advantage must be at most
`a × (1 − 0.40)`. This is a comparison of attack advantage, not ordinary
classification accuracy.

In [ ]:
health = requests.get(f'{BASE_URL}/health', timeout=30).json()
baseline = requests.get(f'{BASE_URL}/baseline', timeout=30).json()
print('Health:', health)
print('Baseline:', baseline)
BASELINE_MIA = float(baseline['baseline_mia_advantage'])
MAX_ALLOWED_MIA = BASELINE_MIA * (1.0 - float(baseline['required_improvement']))
print(f'Maximum allowed defended MIA advantage: {MAX_ALLOWED_MIA:.4f}')

## 3. Dataset and exact validator architecture

The required normalization is mean `0.2860` and standard deviation `0.3530`.
The code below reproduces the supplied architecture exactly: three
convolution layers, max pooling after each one, then the specified two linear
layers. Privacy defenses must change training behavior, not the saved model
shape or parameter names.

In [ ]:
# Define torchvision's optional NMS operator before importing datasets on some Windows builds.
try:
    _tv_compat = torch.library.Library('torchvision', 'DEF')
    _tv_compat.define('nms(Tensor boxes, Tensor scores, float iou_threshold) -> Tensor')
except Exception:
    pass
from torchvision import datasets, transforms

NORMALIZE = transforms.Normalize((0.2860,), (0.3530,))
transform = transforms.Compose([transforms.ToTensor(), NORMALIZE])
train_dataset = datasets.FashionMNIST('AI_Privacy/data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST('AI_Privacy/data', train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=512, shuffle=False, num_workers=0, pin_memory=False)

class FashionMNISTCNN(nn.Module):
    def __init__(self):
        super(FashionMNISTCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(64 * 3 * 3, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = x.view(-1, 64 * 3 * 3)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

model = FashionMNISTCNN().to(DEVICE)
print('Train/test sizes:', len(train_dataset), len(test_dataset))

## 4. Privacy-oriented training

Three choices reduce memorization: input noise makes each training view
slightly different, label smoothing prevents extremely confident one-hot
targets, and AdamW weight decay penalizes unnecessarily large weights.
Together they trade a little utility for a lower membership signal while
leaving the required architecture unchanged.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        # Formula implemented here: x_tilde = x + Normal(0, sigma^2).
        noisy_images = images + INPUT_NOISE_STD * torch.randn_like(images)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(noisy_images), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        running_loss += loss.item() * len(images)
    print(f'Epoch {epoch + 1}/{EPOCHS}; loss={running_loss / len(train_dataset):.4f}')

save_file({key: value.detach().cpu() for key, value in model.state_dict().items()}, str(MODEL_PATH))
print('Saved:', MODEL_PATH)

## 5. Local utility and membership-inference checks

A simple confidence attack uses the model's maximum softmax probability.
Members often receive higher confidence than non-members. We report the
largest threshold-separation advantage as a local warning signal; the server
remains authoritative.

In [ ]:
def confidence_scores(loader):
    model.eval()
    scores, correct, total = [], 0, 0
    with torch.no_grad():
        for images, labels in loader:
            logits = model(images.to(DEVICE))
            probabilities = logits.softmax(dim=1)
            scores.extend(probabilities.max(dim=1).values.cpu().numpy())
            correct += (logits.argmax(dim=1).cpu() == labels).sum().item()
            total += len(labels)
    return np.asarray(scores), correct / total

# A non-shuffled loader is needed only so scores line up with labels; membership
# advantage itself uses the score distributions, not the order.
train_eval_loader = torch.utils.data.DataLoader(train_dataset, batch_size=512, shuffle=False, num_workers=0)
member_scores, train_accuracy = confidence_scores(train_eval_loader)
nonmember_scores, test_accuracy = confidence_scores(test_loader)
thresholds = np.unique(np.concatenate([member_scores, nonmember_scores]))
mia_accuracy = max(0.5 * (member_scores >= t).mean() + 0.5 * (nonmember_scores < t).mean() for t in thresholds)
mia_advantage = float(mia_accuracy - 0.5)
print(f'Train accuracy: {train_accuracy:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')
print(f'Local MIA advantage: {mia_advantage:.4f}')
print(f'Local accuracy requirement met: {test_accuracy >= 0.70}')

## 6. Submit only after local checks

The uploaded artifact contains only the required safetensors state dictionary.
Submission is guarded by `SUBMIT`; it is intentionally false in the source
notebook so rerunning it cannot accidentally submit.

In [ ]:
if SUBMIT:
    if test_accuracy < 0.70:
        raise RuntimeError('Refusing submission: local test accuracy is below 70%.')
    with MODEL_PATH.open('rb') as model_file:
        response = requests.post(f'{BASE_URL}/submit', files={'defended_model': ('defended_model.safetensors', model_file, 'application/octet-stream')}, timeout=REQUEST_TIMEOUT)
    print(response.status_code, response.text)
else:
    print('Submission disabled. Set SUBMIT = True only after reviewing local metrics.')